## Purpose
Build an AI-ready machine learning dataset for predicting aboveground carbon density (AGC) in Central Sulawesi, Indonesia.

## Repo Structure
```text
ForestLUC/
│
├── data/
│
├── metadata/
│   ├── provenance.yaml
│   ├── variables.yaml
│   └── dataset_card.yaml
│
├── reports/
│
├── Notebooks/
│   └── AI_ready_data.ipynb
│
├── requirements.txt
│
└── .gitignore
```

### Directory Descriptions

- `data/` : Final AI-ready dataset exports.
- `metadata/` : Dataset documentation and provenance.
- `reports/` : AI-readiness diagnostics and summary reports.
- `Notebooks/` : Reproducible dataset generation workflow.
- `requirements.txt` : Python dependencies required to recreate the environment.
- `.gitignore` : Files and directories excluded from version control.

In [72]:
# Importing required libraries  
import ee
import yaml
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.decomposition import PCA

In [73]:
# Earth engine initialization
ee.Initialize(
    project="esoteric-realm-483817-u5"
)

print("Earth Engine initialized")

Earth Engine initialized


In [ ]:
# Dataset configuration

TARGET_SAMPLE_SIZE = 1000 # define the target sample size for the dataset 

PREDICTOR_YEAR = 2010

SPLIT_SEED = 42 

RANDOM_SAMPLE_SEED = 42

OUTPUT_FILE = "../data/observations.parquet"

QUALITY_REPORT_FILE = "../reports/quality_report.yaml"

DISTRIBUTION_REPORT_FILE = "../reports/distribution_report.yaml"

DIMENSIONALITY_REPORT_FILE = "../reports/dimensionality_report.yaml"

In [75]:
# Define the region of interest (ROI) using the GAUL dataset
gaul = ee.FeatureCollection(
    "FAO/GAUL_SIMPLIFIED_500m/2025/level1"
)

roi = (
    gaul
    .filter(ee.Filter.eq("GAUL0_NAME", "Indonesia"))
    .filter(ee.Filter.eq("GAUL1_NAME", "Sulawesi Tengah"))
)

roi_geom = roi.geometry()

print(
    "ROI features:",
    roi.size().getInfo()
)

feature = roi.first()

print(
    "Country:",
    feature.get("GAUL0_NAME").getInfo()
)

print(
    "Province:",
    feature.get("GAUL1_NAME").getInfo()
)

ROI features: 1
Country: Indonesia
Province: Sulawesi Tengah


In [76]:
# Target dataset configuration
agc = (
    ee.ImageCollection(
        "NASA/ORNL/biomass_carbon_density/v1"
    )
    .select([
        "agb",
        "agb_uncertainty"
    ])
    .first()
    .clipToCollection(roi)
)

In [77]:
# Predictor builder
def build_predictors(
    start_date,
    end_date
):

# MOD13Q1
    vi = (
        ee.ImageCollection(
            "MODIS/061/MOD13Q1"
        )
        .select([
            "NDVI",
            "EVI",
            "DetailedQA",
            "SummaryQA"
        ])
        .filterDate(
            start_date,
            end_date
        )
        .median()
    )

# MOD15A2H 
    lai_fpar = (
        ee.ImageCollection(
            "MODIS/061/MOD15A2H"
        )
        .select([
            "Lai_500m",
            "Fpar_500m",
            "FparLai_QC"
        ])
        .filterDate(
            start_date,
            end_date
        )
        .median()
    )    

# Landcover
    landcover = (
        ee.ImageCollection(
            "MODIS/061/MCD12Q1"
        )
        .select("LC_Type1")
        .filterDate(
            start_date,
            end_date
        )
        .mode()
    )

# Elevation 
    elevation = (
        ee.Image("USGS/GTOPO30")
        .select("elevation")
    )   

# WHRC 
    whrc = (
        ee.Image(
            "WHRC/biomass/tropical"
        )
        .select("Mg")
    )

# Return predictor stack
    return ee.Image.cat([
        vi,
        lai_fpar,
        landcover,
        elevation
    ])

# Build prectictors for the specified year
predictors = build_predictors(
    "2010-01-01",
    "2011-01-01"
)

In [78]:
# Create final feature stack 
dataset_image = (
    agc
    .addBands(predictors)
)

In [79]:
# Random sampling 
samples = dataset_image.sample(
    region=roi_geom,
    scale=500,
    numPixels=TARGET_SAMPLE_SIZE,
    seed=RANDOM_SAMPLE_SEED,
    dropNulls=True,
    geometries=True,
    tileScale=4
)

print(
    "Requested sample size:",
    TARGET_SAMPLE_SIZE
)

Requested sample size: 1000


In [80]:
# Export to Dataframe
features = samples.getInfo()["features"]

rows = []

for f in features:

    props = f["properties"]

    coords = f["geometry"]["coordinates"]

    props["longitude"] = coords[0]
    props["latitude"] = coords[1]

    rows.append(props)

df = pd.DataFrame(rows)

print(df.shape)

(974, 13)


In [81]:
# Clean column names
df = df.rename(
    columns={
        "agb": "agb_carbon_mg_ha",
        "agb_uncertainty": "agb_uncertainty_mg_ha",
        "NDVI": "ndvi",
        "EVI": "evi",
        "Lai_500m": "lai_m2_m2",
        "Fpar_500m": "fpar_fraction",
        "LC_Type1": "landcover_igbp",
        "Mg": "whrc_live_woody_biomass_mg_ha",
        "elevation": "elevation_m",
        "DetailedQA": "mod13_detailed_qa",
        "SummaryQA": "mod13_summary_qa",
        "FparLai_QC": "mod15_fparlai_qc"
    }
)

In [82]:
# Apply scale factors
df["ndvi"] *= 0.0001
df["evi"] *= 0.0001

df["lai_m2_m2"] *= 0.1
df["fpar_fraction"] *= 0.01

In [83]:
# Create spatial blocks
BLOCK_SIZE = 0.2

df["grid_col"] = (
    df["longitude"] / BLOCK_SIZE
).astype(int)

df["grid_row"] = (
    df["latitude"] / BLOCK_SIZE
).astype(int)

df["spatial_block_id"] = (
    df["grid_col"].astype(str)
    + "_"
    + df["grid_row"].astype(str)
)

In [84]:
# Create data splits
blocks = (
    df["spatial_block_id"]
    .drop_duplicates()
    .sample(
        frac=1,
        random_state=SPLIT_SEED
    )
)

n_blocks = len(blocks)

train_end = int(
    0.70 * n_blocks
)

val_end = int(
    0.85 * n_blocks
)

train_blocks = set(
    blocks.iloc[:train_end]
)

val_blocks = set(
    blocks.iloc[train_end:val_end]
)

test_blocks = set(
    blocks.iloc[val_end:]
)

def assign_split(block):

    if block in train_blocks:
        return "train"

    if block in val_blocks:
        return "validation"

    return "test"


df["split"] = (
    df["spatial_block_id"]
    .apply(assign_split)
)

In [85]:
# Export dataset
df.to_parquet(
    OUTPUT_FILE,
    index=False
)

print(
    "Saved:",
    OUTPUT_FILE
)

Saved: ../data/observations.parquet


In [89]:
# Checking the dataset
df = pd.read_parquet("../data/observations.parquet")

print(df.shape)
print(df.columns.tolist())

df.sample(5)

(974, 17)
['mod13_detailed_qa', 'evi', 'mod15_fparlai_qc', 'fpar_fraction', 'landcover_igbp', 'lai_m2_m2', 'ndvi', 'mod13_summary_qa', 'agb_carbon_mg_ha', 'agb_uncertainty_mg_ha', 'elevation_m', 'longitude', 'latitude', 'grid_col', 'grid_row', 'spatial_block_id', 'split']


,mod13_detailed_qa,evi,mod15_fparlai_qc,fpar_fraction,landcover_igbp,lai_m2_m2,ndvi,mod13_summary_qa,agb_carbon_mg_ha,agb_uncertainty_mg_ha,elevation_m,longitude,latitude,grid_col,grid_row,spatial_block_id,split
86,2116,0.5225,32.0,0.890,2,5.90,0.7842,0,53.054600,21.015289,7,121.429417,1.121938,607,5,607_5,train
376,2116,0.6239,97.0,0.885,2,5.75,0.8464,1,108.092606,38.910213,1,123.528920,-1.289376,617,-6,617_-6,train
100,2116,0.6052,97.0,0.800,2,3.95,0.8165,1,120.260902,43.500385,256,119.944198,0.181237,599,0,599_0,train
341,2120,0.5550,32.0,0.880,2,5.40,0.8303,1,146.496002,44.840282,1443,120.034232,-1.301196,600,-6,600_-6,test
679,2120,0.3129,101.0,0.310,2,0.75,0.6008,3,95.276001,33.701107,2386,119.842336,-1.418351,599,-7,599_-7,train


In [ ]:
# Dataframe metadata 
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   mod13_detailed_qa      974 non-null    int64  
 1   evi                    974 non-null    float64
 2   mod15_fparlai_qc       974 non-null    float64
 3   fpar_fraction          974 non-null    float64
 4   landcover_igbp         974 non-null    int64  
 5   lai_m2_m2              974 non-null    float64
 6   ndvi                   974 non-null    float64
 7   mod13_summary_qa       974 non-null    int64  
 8   agb_carbon_mg_ha       974 non-null    float64
 9   agb_uncertainty_mg_ha  974 non-null    float64
 10  elevation_m            974 non-null    int64  
 11  longitude              974 non-null    float64
 12  latitude               974 non-null    float64
 13  grid_col               974 non-null    int64  
 14  grid_row               974 non-null    int64  
 15  spatial_block_id 

In [ ]:
# Summary statistics of dataframe
df.describe()

,mod13_detailed_qa,evi,mod15_fparlai_qc,fpar_fraction,landcover_igbp,lai_m2_m2,ndvi,mod13_summary_qa,agb_carbon_mg_ha,agb_uncertainty_mg_ha,elevation_m,longitude,latitude,grid_col,grid_row
count,974.000000,974.000000,974.000000,974.000000,974.000000,974.000000,974.000000,974.000000,974.000000,974.000000,974.000000,974.000000,974.000000,974.000000,974.000000
mean,2712.942505,0.507544,50.841889,0.762295,3.165298,4.374487,0.782970,1.188912,96.360258,33.108590,626.703285,121.202174,-1.004051,605.503080,-4.735113
std,3853.945644,0.094963,31.961905,0.188437,2.644855,1.970567,0.101349,1.089632,33.811333,19.010054,506.496764,0.991617,1.084186,4.949227,5.069269
min,2062.000000,0.165000,0.000000,0.110000,2.000000,0.200000,0.225300,0.000000,0.640000,0.000000,1.000000,119.497790,-3.598855,597.000000,-17.000000
25%,2116.000000,0.450525,32.000000,0.671250,2.000000,2.600000,0.748550,0.000000,78.289549,25.961384,220.000000,120.343706,-1.678075,601.000000,-8.000000
50%,2120.000000,0.521800,32.000000,0.870000,2.000000,5.400000,0.818850,1.000000,98.550549,38.240063,535.000000,121.164803,-1.265940,605.000000,-6.000000
75%,2376.000000,0.580650,97.000000,0.890000,2.000000,6.000000,0.851675,1.000000,117.222322,44.865033,919.500000,121.896696,-0.525733,609.000000,-2.000000
max,51199.000000,0.716900,105.000000,0.930000,14.000000,6.500000,0.908300,3.000000,232.246811,88.750107,2912.000000,123.868572,1.308507,619.000000,6.000000


In [ ]:
# Checking for missing values
df.isna().sum()

mod13_detailed_qa        0
evi                      0
mod15_fparlai_qc         0
fpar_fraction            0
landcover_igbp           0
lai_m2_m2                0
ndvi                     0
mod13_summary_qa         0
agb_carbon_mg_ha         0
agb_uncertainty_mg_ha    0
elevation_m              0
longitude                0
latitude                 0
grid_col                 0
grid_row                 0
spatial_block_id         0
split                    0
dtype: int64

In [93]:
# Checking split counts
df["split"].value_counts()

split
train         693
test          145
validation    136
Name: count, dtype: int64

In [ ]:
# QUALITY REPORT 
missing_report = {}

for col in df.columns:

    missing_report[col] = {
        "count": int(
            df[col].isna().sum()
        ),

        "percent": float(
            100 *
            df[col].isna().mean()
        )
    }

sentinel_report = {}

for col in df.columns:

    sentinel_report[col] = {

        "-9999":
        int((df[col] == -9999).sum())
        if pd.api.types.is_numeric_dtype(df[col])
        else 0,

        "-999":
        int((df[col] == -999).sum())
        if pd.api.types.is_numeric_dtype(df[col])
        else 0
    }

unc = df[
    "agb_uncertainty_mg_ha"
]

uncertainty_summary = {

    "mean":
    float(unc.mean()),

    "median":
    float(unc.median()),

    "std":
    float(unc.std()),

    "p05":
    float(
        unc.quantile(0.05)
    ),

    "p95":
    float(
        unc.quantile(0.95)
    ),

    "max":
    float(
        unc.max()
    )
}

qa_summary = {

    "mod13_summary_qa":
    df["mod13_summary_qa"]
      .value_counts()
      .to_dict(),

    "mod15_fparlai_qc":
    df["mod15_fparlai_qc"]
      .value_counts()
      .to_dict()
}

quality_report = {

    "dataset_size":
    len(df),

    "missing_values":
    missing_report,

    "sentinel_values":
    sentinel_report,

    "uncertainty_summary":
    uncertainty_summary,

    "quality_flags":
    qa_summary
}

with open(
    "reports/quality_report.yaml",
    "w"
) as f:

    yaml.dump(
        quality_report,
        f,
        sort_keys=False
    )


FileNotFoundError: [Errno 2] No such file or directory: 'reports/quality_report.yaml'

In [104]:
# DISTRIBUTION REPORT 
distribution_report = {}

for col in [

    "agb_carbon_mg_ha",
    "ndvi",
    "evi",
    "lai_m2_m2",
    "fpar_fraction",
    "elevation_m"
]:

    distribution_report[col] = {

        "min":
        float(df[col].min()),

        "p05":
        float(df[col].quantile(0.05)),

        "median":
        float(df[col].median()),

        "p95":
        float(df[col].quantile(0.95)),

        "max":
        float(df[col].max())
    }

distribution_report[
    "landcover_distribution"
] = (
    df["landcover_igbp"]
    .value_counts()
    .to_dict()
)

with open(
    "reports/distribution_report.yaml",
    "w"
) as f:

    yaml.dump(
        distribution_report,
        f,
        sort_keys=False
    )

FileNotFoundError: [Errno 2] No such file or directory: 'reports/distribution_report.yaml'

In [ ]:
# DIMENSIONALITY REPORT
predictors = [

    "ndvi",
    "evi",
    "lai_m2_m2",
    "fpar_fraction",
    "landcover_igbp",
    "elevation_m",
    "whrc_live_woody_biomass_mg_ha"
]

X = (
    df[predictors]
    .fillna(
        df[predictors].median()
    )
)

pca = PCA()

pca.fit(X)

dimensionality_report = {

    "N":
    int(len(df)),

    "D":
    int(len(predictors)),

    "N_over_D":
    float(
        len(df) /
        len(predictors)
    ),

    "variance_explained":

    pca.explained_variance_ratio_
       .tolist()
}

with open(
    "reports/dimensionality_report.yaml",
    "w"
) as f:

    yaml.dump(
        dimensionality_report,
        f,
        sort_keys=False
    )